### Module 6: Core RAG – Concept

In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

True

Step 2: Load Vector Store and Create Retriever

In [2]:
# Define the path to the persisted Chroma vector store.
chroma_storage_dir = Path('../../../chroma_db')

# Check if the folder exists
if not chroma_storage_dir.exists():
    print(f'Vector store not found at {chroma_storage_dir}')
    
else:
    print(f'Vector store found at {chroma_storage_dir}')
    
# Create the embedding model 
embeddings = OpenAIEmbeddings()

# Load the existing vector store from disk
vectorstore = Chroma(
    persist_directory=str(chroma_storage_dir),
    embedding_function=embeddings
)

# Create a retriever
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

print('Vector store loaded and retriever created')
retriever

Vector store found at ..\..\..\chroma_db


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store loaded and retriever created


VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001F5FB6DAE10>, search_kwargs={'k': 4})

Step 3: Test the Retriever

In [3]:
# Define a test question
question = 'What are common crop diseases and how can they be controlled?'

# Retrieve the top 4 chunks without any deduplication
raw_docs = retriever.invoke(question)

print(f'Retrieved {len(raw_docs)} chunks:\n')

for i, doc in enumerate(raw_docs, start=1):
    print(f'Chunk {i}')
    print(doc.page_content)
    print('-' * 70)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved 4 chunks:

Chunk 1
Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infected plants early.
----------------------------------------------------------------------
Chunk 2
Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infected plants early.
----------------------------------------------------------------------
Chunk 3
Common Crop Diseases and Control

1. Cassava Mosaic Disease
   Affected crop: Cassava
   Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
   Control: Use disease-free cuttings, plant resistant varieties, and remove infected plants.
-----------------------

Step 4: Deduplicate Retrieved Chunks


In [4]:
def deduplicate_docs(docs):
    '''Return only documents with unique page_content.'''
    
    seen = set()
    unique_docs = []
    
    for doc in docs:
        text = doc.page_content.strip()
        if text not in seen:
            seen.add(text)
            unique_docs.append(doc)
            
    return unique_docs

unique_docs = deduplicate_docs(raw_docs)
print(f'Before deduplication: {len(raw_docs)} chunks')
print(f'After deduplication: {len(unique_docs)} chunks')

Before deduplication: 4 chunks
After deduplication: 2 chunks


Step 5: Define format_docs and Create Prompt

In [5]:
def format_docs(docs):
    '''
    Combine a list of documents into a single context string.
    '''
    
    return '\n\n'.join(doc.page_content for doc in docs)
    
    
# Create the RAG prompt template
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Answer the question using only the provided context. If you don\'t know, say you don\'t know.'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}')
])

print('RAG prompt template created.')

RAG prompt template created.


Step 6: Create LLM and Build RAG Chain

In [6]:
# Create the LLM (generator)
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Build the RAG chain with deduplication
rag_chain = (
    {
        'context': retriever | deduplicate_docs | format_docs,
        'question': RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
print('RAG chain created with deduplication.')

RAG chain created with deduplication.


Step 7: Test the RAG Chain

In [7]:
# Ask the same question
answer = rag_chain.invoke(question)

print('Answer:')
print(answer)

Answer:
One common crop disease is Cassava Mosaic Disease, which affects cassava. The symptoms include yellowing and mottling of leaves, stunted growth, and reduced yield. Control measures include using disease-free cuttings, planting resistant varieties, and removing infected plants early.
